# 03 - Evaluation and Reranking

Measuring how well retrieval works on a test set of questions, then comparing dense search, BM25, hybrid search and a reranker. Part 2 checks retrieval with an LLM judge and measures final answer accuracy.

Runtime: T4 GPU if available (the reranker is about 10x faster), CPU also works.

## Setup

In [1]:
# Installing libraries
!pip install -q chromadb sentence-transformers google-genai rank_bm25

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/13

In [2]:
# Mounting Drive and loading the saved index
from google.colab import drive
drive.mount('/content/drive')

import json, re, time, random, shutil
from pathlib import Path
import numpy as np, pandas as pd
import torch, chromadb
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from IPython.display import display

PROJECT_DIR = Path("/content/drive/MyDrive/annual-report-rag")
EVAL_DIR = PROJECT_DIR / "eval"
EVAL_DIR.mkdir(exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# loading the docstore (original text, tables, image paths)
docstore = json.loads((PROJECT_DIR / "docstore.json").read_text())

# copying Chroma from Drive to local disk and opening it
CHROMA_LOCAL = f"/content/chroma_{int(time.time())}"
shutil.copytree(PROJECT_DIR / "chroma", CHROMA_LOCAL)
col = chromadb.PersistentClient(path=CHROMA_LOCAL).get_collection("annual_reports")

# loading the embedding model
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)
print(f"Chroma: {col.count()} items | docstore: {len(docstore)} items")

Mounted at /content/drive
device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chroma: 4059 items | docstore: 4059 items


In [3]:
# Parsing company and year from the question, and running dense search
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
COMPANIES = sorted({d["company"] for d in docstore.values()})
YEARS = sorted({d["year"] for d in docstore.values()})
ALIASES = {c: [c.replace("_", " ")] for c in COMPANIES}
ALIASES.setdefault("au_small_finance_bank", []).extend(["au bank", "au sfb"])

def find_years(q):
    ys, q = set(), q.lower()
    # "2023-24" -> 2024
    for m in re.finditer(r"(20\d{2})\s*[-–/]\s*(\d{2,4})", q):
        end = m.group(2); ys.add(end if len(end) == 4 else "20" + end)
    q = re.sub(r"(20\d{2})\s*[-–/]\s*(\d{2,4})", " ", q)
    # "FY24" -> 2024
    for m in re.finditer(r"fy\s*'?(\d{2,4})\b", q):
        y = m.group(1); ys.add(y if len(y) == 4 else "20" + y)
    q = re.sub(r"fy\s*'?(\d{2,4})\b", " ", q)
    # plain "2024"
    ys.update(re.findall(r"\b(20\d{2})\b", q))
    return sorted(y for y in ys if y in YEARS)

def parse_query(q):
    ql = q.lower()
    company = next((c for c, names in ALIASES.items() if any(n in ql for n in names)), None)
    return company, find_years(q)

def make_where(company, years):
    conds = []
    if company: conds.append({"company": company})
    if years:   conds.append({"year": years[0]} if len(years) == 1 else {"year": {"$in": years}})
    return None if not conds else conds[0] if len(conds) == 1 else {"$and": conds}

def dense_search(q, n=30, company=None, years=None):
    qv = embedder.encode([QUERY_PREFIX + q], normalize_embeddings=True)[0].tolist()
    res = col.query(query_embeddings=[qv], n_results=n, where=make_where(company, years))
    return res["ids"][0]

print(parse_query("gross NPA in FY 2023-24"), dense_search("gross NPA closing balance", n=3))

(None, ['2024']) ['au_small_finance_bank_2025_p121_table_42', 'au_small_finance_bank_2024_p121_table_54', 'au_small_finance_bank_2024_p121_table_52']


In [4]:
# Connecting to Gemini (pinned to one model, no fallback during evaluation)
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
QGEN_MODEL = "gemini-3.5-flash-lite"

def gemini_json(contents, model=QGEN_MODEL, retries=5):
    for a in range(retries):
        try:
            r = client.models.generate_content(
                model=model, contents=contents,
                config=types.GenerateContentConfig(temperature=0, response_mime_type="application/json"))
            return json.loads(r.text)
        except json.JSONDecodeError:
            time.sleep(3)
        except Exception as ex:
            m = str(ex)
            if "503" in m or "UNAVAILABLE" in m:
                print(f"   overloaded, waiting {5 * 2 ** a}s"); time.sleep(5 * 2 ** a)
            elif "429" in m or "RESOURCE_EXHAUSTED" in m:
                if "PerDay" in m:
                    raise RuntimeError(f"Daily quota reached for {model}, rerun tomorrow")
                time.sleep(30)
            else:
                raise
    return None

## Building the golden test set

In [5]:
# Picking source items for the test questions (fixed seed, same sample every run)
random.seed(42)

pools = {
    "table": [d for d in docstore.values() if d["type"] == "table" and d.get("summary")
              and d["summary"] != "SKIP" and d.get("shape", [0, 0])[0] >= 3],
    "text":  [d for d in docstore.values() if d["type"] == "text" and len(d["content"]) > 400],
    "image": [d for d in docstore.values() if d["type"] == "image" and d.get("summary")
              and d["summary"] != "SKIP"],
}
PLAN = {"table": 24, "text": 16, "image": 10}

candidates = []
for kind, n in PLAN.items():
    picked = random.sample(pools[kind], min(n + 6, len(pools[kind])))   # 6 spare in case of SKIP
    candidates += [(kind, d) for d in picked]
    print(f"{kind}: pool {len(pools[kind])}, picked {len(picked)}")

table: pool 641, picked 30
text: pool 1866, picked 22
image: pool 130, picked 16


In [6]:
# Generating one question per item with Gemini (cached on Drive)
QGEN_CACHE = EVAL_DIR / "qgen_cache.json"
qgen = json.loads(QGEN_CACHE.read_text()) if QGEN_CACHE.exists() else {}

QGEN_PROMPT = """You create evaluation questions for a search system over Indian bank annual reports.
For EACH item below, write ONE question that:
- a financial analyst might realistically ask,
- can be answered fully from this item alone,
- is about the report's own fiscal year (given in the item header), and names that year
  (e.g. "FY 2023-24" or "FY24"),
- does NOT mention "this table", "this chart", "the item", or page numbers,
- paraphrases rather than copying long phrases from the item.
Also give the short correct answer, with exact numbers and units where relevant.
If an item can't support a good question, set the question to "SKIP".
Return ONLY a JSON list: [{"id": "...", "question": "...", "answer": "..."}]"""

def header(d):
    y = int(d["year"])
    return f"ITEM {d['id']} | {d['company'].replace('_', ' ')} annual report, FY {y - 1}-{str(y)[2:]} | type: {d['type']}"

def body(d):
    if d["type"] == "table": return d["content"][:4000]
    if d["type"] == "image": return f"Chart/infographic. Caption: {d.get('caption', '')}\n{d['summary']}"
    return d["content"][:3000]

todo = [d for _, d in candidates if d["id"] not in qgen]
print(f"{len(todo)} items need questions")
for i in range(0, len(todo), 5):
    batch = todo[i:i + 5]
    out = gemini_json([QGEN_PROMPT] + [f"{header(d)}\n{body(d)}" for d in batch])
    if isinstance(out, dict): out = out.get("items", [out])
    for r in out or []:
        if isinstance(r, dict) and r.get("id") in docstore:
            qgen[r["id"]] = {"question": r.get("question", "SKIP"), "answer": r.get("answer", "")}
    QGEN_CACHE.write_text(json.dumps(qgen, ensure_ascii=False, indent=1))
    print(f"   {len(qgen)} done")
    time.sleep(4)

0 items need questions


In [7]:
# Assembling the golden set: generated + hand-written + unanswerable questions
rows = []
for kind, n in PLAN.items():
    good = [d for k, d in candidates
            if k == kind and qgen.get(d["id"], {}).get("question", "SKIP") != "SKIP"][:n]
    for d in good:
        rows.append({"qtype": kind, "question": qgen[d["id"]]["question"],
                     "answer": qgen[d["id"]]["answer"], "gold_doc_id": d["doc_id"],
                     "gold_pages": [d["page"]], "source_id": d["id"]})

# adding hand-written questions from Notebook 2's tests
HAND = [
    ("What was the gross NPA closing balance in FY 2023-24?", "₹1,237.40 crore",
     "au_small_finance_bank_2024", [148]),
    ("How did total advances change between FY23 and FY24?", "₹59,158 cr → ₹73,999 cr (+25.09%)",
     "au_small_finance_bank_2024", [54]),
    ("What are the key risks highlighted by management in FY2025?",
     "Credit, operational, market, liquidity, fraud, compliance, cyber; emerging: climate, data privacy",
     "au_small_finance_bank_2025", [22, 74]),
    ("Show me the business segments of AU Small Finance Bank in FY2023",
     "10 verticals incl. Branch Banking, Wheels, SBL, Housing Loans...",
     "au_small_finance_bank_2023", [179, 4]),
]
for q, a, doc, pages in HAND:
    rows.append({"qtype": "hand", "question": q, "answer": a, "gold_doc_id": doc,
                 "gold_pages": pages, "source_id": ""})

# adding unanswerable questions (the system should refuse these)
for q in ["Who is the CEO of Google?",
          "What was Infosys's revenue in FY2024?",
          "What is the bank's market share in the US credit card market?",
          "How many branches does HDFC Bank have?",
          "What will the bank's net profit be in FY2030?"]:
    rows.append({"qtype": "unanswerable", "question": q, "answer": "NOT FOUND",
                 "gold_doc_id": "", "gold_pages": [], "source_id": ""})

golden = pd.DataFrame(rows)
golden.insert(0, "qid", [f"q{i:03d}" for i in range(len(golden))])
print(golden.qtype.value_counts())

pd.set_option("display.max_colwidth", 160)
display(golden[["qid", "qtype", "question", "answer", "gold_pages"]])

qtype
table           24
text            16
image           10
unanswerable     5
hand             4
Name: count, dtype: int64


,qid,qtype,question,answer,gold_pages
0,q000,table,"What was the net block value of depreciation and fixed assets for AU Small Finance Bank as of March 31, 2023 for FY 2022-23?","6,482,877 (in '000)",[229]
1,q001,table,Who served as the Chairperson of the committee containing independent directors in AU Small Finance Bank during FY 2022-23?,Mr. M. S. Sriram,[137]
2,q002,table,What percentage of employees were covered under Provident Fund (PF) for AU Small Finance Bank in FY 2023-24?,95%,[110]
3,q003,table,What was the unclaimed dividend amount for FY 2017-18 for AU Small Finance Bank as reported for FY 2023-24?,"23,970.50",[94]
4,q004,table,What was the fixed pay including perquisites for the MD & CEO Mr. Sanjay Agarwal of AU Small Finance Bank for FY 2023-24?,2.56 Crore,[86]
5,q005,table,What was the total reduction in gross non-performing advances for FY 2022-23?,"2,020.57 crore",[261]
6,q006,table,"What was the total paid-up equity share capital of the bank as of March 31, 2023, for FY 2022-23?","6,667,450 thousand",[225]
7,q007,table,What was the weighted value of total regulatory capital under ASF items for FY 2024-25?,"12,305.65 ₹ in Crore",[144]
8,q008,table,What was the mean pay of the bank as a whole excluding sub-staff for FY 2024-25?,₹ 0.054 crore,[160]
9,q009,table,What was the total number of unique touchpoints recorded by AU Small Finance Bank in March 2025 for FY 2024-25?,"1,683",[31]


In [8]:
# Dropping ambiguous questions ("the committee" without naming it) and saving the golden set
DROP = ["q001", "q010", "q011"]
golden = golden[~golden.qid.isin(DROP)].reset_index(drop=True)
answerable = golden[golden.qtype != "unanswerable"]

golden.assign(gold_pages=golden.gold_pages.map(lambda p: ";".join(map(str, p)))) \
      .to_csv(EVAL_DIR / "golden.csv", index=False)
print(f"{len(golden)} questions ({len(answerable)} answerable)")

56 questions (51 answerable)


## Retrieval methods

In [9]:
# Defining the scoring functions (Hit@k and MRR, matched on report + page)
COLS = ["hit@1", "hit@3", "hit@5", "hit@10", "mrr"]

def is_gold(hit_id, row):
    d = docstore[hit_id]
    return d["doc_id"] == row.gold_doc_id and d["page"] in row.gold_pages

def score(ranked_ids, row):
    first = next((r for r, i in enumerate(ranked_ids[:10], 1) if is_gold(i, row)), None)
    out = {f"hit@{k}": int(first is not None and first <= k) for k in (1, 3, 5, 10)}
    out["mrr"] = 1 / first if first else 0.0
    return out

def evaluate(retriever, name):
    recs = []
    for row in tqdm(list(answerable.itertuples()), desc=name, leave=False):
        recs.append({"qid": row.qid, "qtype": row.qtype, "variant": name,
                     **score(retriever(row.question), row)})
    return pd.DataFrame(recs)

In [10]:
# Creating the reranker, with scores cached on Drive so they survive restarts
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-base", device=device, max_length=512)

# getting the embedded text of every item (what the reranker reads)
_all = col.get(include=["documents"])
doc_text = dict(zip(_all["ids"], _all["documents"]))

# loading reranker scores saved in earlier runs
RR_FILE = EVAL_DIR / "rerank_cache.json"
_rr_cache = {}
if RR_FILE.exists():
    _rr_cache = {tuple(k.split("|||", 1)): v for k, v in json.loads(RR_FILE.read_text()).items()}
print(f"Loaded {len(_rr_cache)} cached reranker scores")

def rr_scores(q, ids):
    # scoring only the pairs not already in the cache
    need = [i for i in ids if (q, i) not in _rr_cache]
    if need:
        s = reranker.predict([(q, doc_text[i]) for i in need], batch_size=32)
        _rr_cache.update({(q, i): float(v) for i, v in zip(need, s)})
        RR_FILE.write_text(json.dumps({f"{a}|||{b}": v for (a, b), v in _rr_cache.items()}))
    return np.array([_rr_cache[(q, i)] for i in ids])

def rerank(q, ids, top=10, n=20):
    # reranking the top 20 candidates
    ids = ids[:n]
    return [ids[j] for j in np.argsort(-rr_scores(q, ids))[:top]] if ids else []

# checking one example: page order before and after reranking
q = "What was the gross NPA closing balance in FY 2023-24?"
c, y = parse_query(q)
before = dense_search(q, 30, c, y)
print("before:", [docstore[i]["page"] for i in before[:5]])
print("after: ", [docstore[i]["page"] for i in rerank(q, before)[:5]])

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

Loaded 1463 cached reranker scores
before: [149, 148, 148, 54, 122]
after:  [149, 165, 148, 53, 54]


In [11]:
# Building BM25 keyword search and Reciprocal Rank Fusion
from rank_bm25 import BM25Okapi

TOKEN = re.compile(r"[a-z0-9]+(?:[.,][0-9]+)*")       # keeping "1,237.40" and "fy24" whole
def tok(s): return TOKEN.findall(s.lower())

bm_ids = _all["ids"]
bm25 = BM25Okapi([tok(doc_text[i]) for i in bm_ids])

def bm25_search(q, n=30, company=None, years=None):
    scores = bm25.get_scores(tok(q))
    out = []
    for j in np.argsort(-scores):
        d = docstore[bm_ids[j]]
        if company and d["company"] != company: continue
        if years and d["year"] not in years: continue
        out.append(bm_ids[j])
        if len(out) >= n: break
    return out

def rrf(*ranked_lists, k=60, top=30):
    s = {}
    for L in ranked_lists:
        for r, i in enumerate(L, 1):
            s[i] = s.get(i, 0) + 1 / (k + r)
    return [i for i, _ in sorted(s.items(), key=lambda x: -x[1])][:top]

print("BM25 top pages:", [docstore[i]["page"] for i in bm25_search(q, 5, c, y)])

BM25 top pages: [164, 148, 149, 66, 152]


In [12]:
# Defining every retrieval variant to compare
def r_dense_nofilter(q):  return dense_search(q, 10)
def r_dense(q):           c, y = parse_query(q); return dense_search(q, 10, c, y)
def r_bm25(q):            c, y = parse_query(q); return bm25_search(q, 10, c, y)
def r_hybrid(q):          c, y = parse_query(q); return rrf(dense_search(q, 30, c, y), bm25_search(q, 30, c, y))[:10]
def r_dense_rerank(q):    c, y = parse_query(q); return rerank(q, dense_search(q, 30, c, y))
def r_hybrid_rerank(q):   c, y = parse_query(q); return rerank(q, rrf(dense_search(q, 30, c, y), bm25_search(q, 30, c, y)))

def r_dense_rerank_fused(q):
    # blending the dense order and the reranker order instead of replacing one with the other
    c, y = parse_query(q)
    dense = dense_search(q, 30, c, y)[:20]
    reranked = [dense[j] for j in np.argsort(-rr_scores(q, dense))]
    return rrf(dense, reranked, top=10)

VARIANTS = {
    "1. dense, no filters":      r_dense_nofilter,
    "2. dense + filters":        r_dense,              # the Notebook 2 system
    "3. BM25 + filters":         r_bm25,
    "4. hybrid (RRF)":           r_hybrid,
    "5. dense + rerank":         r_dense_rerank,
    "6. hybrid + rerank":        r_hybrid_rerank,
    "7. dense + rerank (fused)": r_dense_rerank_fused,
}

## Ablation table

In [13]:
# Evaluating the fast variants (no reranker)
FAST = {k: v for k, v in VARIANTS.items() if "rerank" not in k}
res_fast = pd.concat([evaluate(f, n) for n, f in FAST.items()], ignore_index=True)
display(res_fast.groupby("variant")[COLS].mean().round(3))

1. dense, no filters:   0%|          | 0/51 [00:00<?, ?it/s]

2. dense + filters:   0%|          | 0/51 [00:00<?, ?it/s]

3. BM25 + filters:   0%|          | 0/51 [00:00<?, ?it/s]

4. hybrid (RRF):   0%|          | 0/51 [00:00<?, ?it/s]

,hit@1,hit@3,hit@5,hit@10,mrr
variant,,,,,
"1. dense, no filters",0.235,0.490,0.529,0.667,0.368
2. dense + filters,0.431,0.588,0.725,0.863,0.554
3. BM25 + filters,0.510,0.667,0.784,0.863,0.618
4. hybrid (RRF),0.588,0.706,0.824,0.863,0.666


In [14]:
# Evaluating the reranker variants (slow the first time on CPU, instant once cached)
SLOW = {k: v for k, v in VARIANTS.items() if "rerank" in k}
t0 = time.time()
res_slow = pd.concat([evaluate(f, n) for n, f in SLOW.items()], ignore_index=True)
print(f"Rerank variants: {(time.time() - t0) / 60:.1f} min | cached pairs: {len(_rr_cache)}")

5. dense + rerank:   0%|          | 0/51 [00:00<?, ?it/s]

6. hybrid + rerank:   0%|          | 0/51 [00:00<?, ?it/s]

7. dense + rerank (fused):   0%|          | 0/51 [00:00<?, ?it/s]

Rerank variants: 0.1 min | cached pairs: 1465


In [15]:
# Combining results into the ablation table and saving it
results = pd.concat([res_fast, res_slow], ignore_index=True)
ablation = results.groupby("variant")[COLS].mean().round(3)

print("    Overall    ")
display(ablation)
print("    Hit@5 by question type    ")
display(results.pivot_table(index="variant", columns="qtype", values="hit@5").round(2))

results.to_csv(EVAL_DIR / "retrieval_results.csv", index=False)
ablation.to_csv(EVAL_DIR / "ablation_retrieval.csv")

    Overall    


,hit@1,hit@3,hit@5,hit@10,mrr
variant,,,,,
"1. dense, no filters",0.235,0.490,0.529,0.667,0.368
2. dense + filters,0.431,0.588,0.725,0.863,0.554
3. BM25 + filters,0.510,0.667,0.784,0.863,0.618
4. hybrid (RRF),0.588,0.706,0.824,0.863,0.666
5. dense + rerank,0.588,0.824,0.863,0.863,0.689
6. hybrid + rerank,0.549,0.765,0.863,0.902,0.660
7. dense + rerank (fused),0.549,0.725,0.843,0.882,0.665


    Hit@5 by question type    


qtype,hand,image,table,text
variant,,,,
"1. dense, no filters",0.25,0.7,0.52,0.50
2. dense + filters,0.75,0.8,0.76,0.62
3. BM25 + filters,0.50,0.6,0.90,0.81
4. hybrid (RRF),0.75,0.7,0.95,0.75
5. dense + rerank,1.00,0.8,0.90,0.81
6. hybrid + rerank,0.75,0.9,0.95,0.75
7. dense + rerank (fused),1.00,0.9,0.86,0.75


In [16]:
# Looking at the questions the best variant still misses
BEST = ablation["hit@5"].idxmax()
print(f"Best variant: {BEST}\n")
misses = results[(results.variant == BEST) & (results["hit@5"] == 0)]
f = VARIANTS[BEST]
for row in golden[golden.qid.isin(misses.qid)].itertuples():
    got = [(docstore[i]["year"], docstore[i]["page"], docstore[i]["type"]) for i in f(row.question)[:3]]
    print(f"[{row.qtype}] {row.question}\n   gold: {row.gold_doc_id} p{row.gold_pages}\n   got:  {got}\n")

Best variant: 5. dense + rerank

[table] What was the total reduction in gross non-performing advances for FY 2022-23?
   gold: au_small_finance_bank_2023 p[261]
   got:  [('2023', 260, 'table'), ('2023', 73, 'text'), ('2023', 268, 'table')]

[table] What was the adjusted profit after tax for AU Small Finance Bank for the year ended March 31, 2024 in FY 2023-24?
   gold: au_small_finance_bank_2024 p[69]
   got:  [('2024', 162, 'table'), ('2024', 126, 'table'), ('2024', 66, 'table')]

[text] What was the maximum ceiling amount approved by the Board for issuing debt securities over and above outstanding debt in FY 2022-23?
   gold: au_small_finance_bank_2023 p[300]
   got:  [('2023', 259, 'table'), ('2023', 259, 'table'), ('2023', 152, 'text')]

[text] Which new skill was added to the Board skill matrix during FY 2023-24 by the Board and Nomination and Remuneration Committee?
   gold: au_small_finance_bank_2024 p[98]
   got:  [('2024', 85, 'text'), ('2024', 112, 'text'), ('2024', 81, 'te

In [17]:
# Inspecting the NPA question: what the reranker ranks above the correct table
q = "What was the gross NPA closing balance in FY 2023-24?"
cands = dense_search(q, 30, *parse_query(q))[:20]
s = rr_scores(q, cands)

print(f"Q: {q}\n")
for i, sc in sorted(zip(cands, s), key=lambda x: -x[1])[:6]:
    d = docstore[i]
    print(f"{sc:.3f} | p{d['page']:<4} {d['type']:5} | {doc_text[i][:250]!r}\n")

gold = "au_small_finance_bank_2024_p121_table_52"      # the correct FY2023-24 table
print("Correct table score:", round(float(s[cands.index(gold)]), 3) if gold in cands else "not in top 20")
print("Fused top 5 pages:", [docstore[i]["page"] for i in r_dense_rerank_fused(q)[:5]])
print("Page-148 items in fused top 5:", [i for i in r_dense_rerank_fused(q)[:5] if docstore[i]["page"] == 148])

Q: What was the gross NPA closing balance in FY 2023-24?

0.941 | p149  text  | 'au small finance bank annual report FY2024, page 149.\nSchedules\n Crore).March 31, 2023 Gross NPAs = 141.96. , (₹ in Crore).% of Gross NPAs to Total Advances in that sector = 1.54%. 4, Sector = Personal loans. 4, (₹ in Crore).Outstanding Total Advance'

0.751 | p165  table | 'Table. au small finance bank annual report FY2024, page 165. \nThis is a financial table detailing the particulars of asset and liability items for AU Small Finance Bank as of March 31, 2024, and March 31, 2023. Key figures include provisions for NPA '

0.727 | p148  table | 'Table. au small finance bank annual report FY2024, page 148. \nThis table presents the movement of gross standard advances, NPAs, and provisions for AU Small Finance Bank for the prior year ended March 31, 2023. Total gross advances and NPAs opened at'

0.599 | p53   table | 'Table. au small finance bank annual report FY2024, page 53. \nThis is the balance sheet

## Refusing unanswerable questions

In [18]:
# Getting the top reranker score for each question
def top_rerank_score(q):
    c, y = parse_query(q)
    return float(rr_scores(q, dense_search(q, 30, c, y)[:20]).max())

golden["top_score"] = golden.question.map(top_rerank_score)
golden["answerable"] = golden.qtype != "unanswerable"
display(golden.groupby("answerable")["top_score"].describe().round(3))
display(golden[~golden.answerable][["question", "top_score"]])

,count,mean,std,min,25%,50%,75%,max
answerable,,,,,,,,
False,5.0,0.386,0.356,0.002,0.028,0.469,0.663,0.77
True,51.0,0.975,0.108,0.268,0.996,0.999,1.000,1.00


,question,top_score
51,Who is the CEO of Google?,0.001651
52,What was Infosys's revenue in FY2024?,0.770015
53,What is the bank's market share in the US credit card market?,0.028398
54,How many branches does HDFC Bank have?,0.663313
55,What will the bank's net profit be in FY2030?,0.468941


In [19]:
# Trying different refusal thresholds
n_ans, n_un = golden.answerable.sum(), (~golden.answerable).sum()
for t in [0.5, 0.7, 0.8, 0.9, 0.95]:
    caught = (~golden.answerable & (golden.top_score < t)).sum()
    wrong  = (golden.answerable & (golden.top_score < t)).sum()
    print(f"threshold {t}: refuses {caught}/{n_un} unanswerable | wrongly refuses {wrong}/{n_ans} answerable")

threshold 0.5: refuses 3/5 unanswerable | wrongly refuses 1/51 answerable
threshold 0.7: refuses 4/5 unanswerable | wrongly refuses 1/51 answerable
threshold 0.8: refuses 5/5 unanswerable | wrongly refuses 2/51 answerable
threshold 0.9: refuses 5/5 unanswerable | wrongly refuses 2/51 answerable
threshold 0.95: refuses 5/5 unanswerable | wrongly refuses 4/51 answerable


# Part 2: LLM-judged relevance and answer accuracy

In [20]:
# Collecting the top 5 results for every variant and question
top5 = {(name, row.qid): f(row.question)[:5]
        for name, f in VARIANTS.items() for row in answerable.itertuples()}
rows_by_qid = {r.qid: r for r in answerable.itertuples()}
print(f"{len(top5)} result lists, "
      f"{len({(q, i) for (v, q), L in top5.items() for i in L})} unique question-passage pairs to judge")

357 result lists, 742 unique question-passage pairs to judge


In [21]:
# Judging each retrieved passage with Gemini: does it actually answer the question? (cached)
JUDGE_MODEL = "gemini-3.5-flash-lite"
REL_CACHE = EVAL_DIR / "relevance_cache.json"
rel = json.loads(REL_CACHE.read_text()) if REL_CACHE.exists() else {}

REL_PROMPT = """You check search results for a question about a bank's annual report.
QUESTION: {q}
REFERENCE ANSWER: {a}
For EACH passage below, decide: does it contain the information needed to answer the question
correctly (consistent with the reference answer, for the right fiscal year)?
Merely mentioning the topic is NOT enough.
Return ONLY a JSON list: [{{"pid": "P1", "relevant": true}}, ...]"""

def passage(i):
    d = docstore[i]
    tag = f"[{d['type']}, FY{d['year']} report, p.{d['page']}]"
    return f"{tag} {d.get('summary', '')}" if d["type"] == "image" else f"{tag} {d['content'][:2500]}"

for row in tqdm(list(answerable.itertuples())):
    ids = sorted({i for (v, qid), L in top5.items() if qid == row.qid for i in L})
    todo = [i for i in ids if f"{row.qid}|{i}" not in rel]
    if not todo:
        continue
    pid = {f"P{k + 1}": i for k, i in enumerate(todo)}
    out = gemini_json([REL_PROMPT.format(q=row.question, a=row.answer)]
                      + [f"{p}: {passage(i)}" for p, i in pid.items()], model=JUDGE_MODEL)
    if isinstance(out, dict):
        out = out.get("items", [out])
    for r in out or []:
        if isinstance(r, dict) and r.get("pid") in pid:
            rel[f"{row.qid}|{pid[r['pid']]}"] = bool(r.get("relevant"))
    REL_CACHE.write_text(json.dumps(rel))
    time.sleep(4)
print(f"{len(rel)} judgments cached")

  0%|          | 0/51 [00:00<?, ?it/s]

841 judgments cached


In [22]:
# Scoring with judged relevance, and checking the judge itself
def judged_ok(qid, i):
    return rel.get(f"{qid}|{i}", False) or is_gold(i, rows_by_qid[qid])

recs = []
for (v, qid), L in top5.items():
    first = next((r for r, i in enumerate(L, 1) if judged_ok(qid, i)), None)
    recs.append({"variant": v, "qid": qid, "qtype": rows_by_qid[qid].qtype,
                 "j_hit@1": int(first == 1), "j_hit@3": int(first is not None and first <= 3),
                 "j_hit@5": int(first is not None), "j_mrr@5": 1 / first if first else 0.0})
judged = pd.DataFrame(recs)

print("   Strict (gold page) vs judged    ")
display(ablation[["hit@5", "mrr"]].join(
    judged.groupby("variant")[["j_hit@1", "j_hit@3", "j_hit@5", "j_mrr@5"]].mean().round(3)))
print("    Judged Hit@5 by type    ")
display(judged.pivot_table(index="variant", columns="qtype", values="j_hit@5").round(2))

# spot-checking the judge: gold items it rejected, and new finds it accepted
pairs = {(qid, i) for (v, qid), L in top5.items() for i in L}
disagree = [(q, i) for q, i in pairs if is_gold(i, rows_by_qid[q]) and rel.get(f"{q}|{i}") is False]
new_finds = [(q, i) for q, i in pairs if rel.get(f"{q}|{i}") and not is_gold(i, rows_by_qid[q])]
print(f"\nGold items judged NOT relevant: {len(disagree)} | non-gold items judged relevant: {len(new_finds)}")
for q, i in random.sample(new_finds, min(4, len(new_finds))):
    print(f"\n[{q}] {rows_by_qid[q].question}\n  ref: {rows_by_qid[q].answer}\n  "
          f"found: p{docstore[i]['page']} → {passage(i)[:300]}")

   Strict (gold page) vs judged    


,hit@5,mrr,j_hit@1,j_hit@3,j_hit@5,j_mrr@5
variant,,,,,,
"1. dense, no filters",0.529,0.368,0.353,0.549,0.667,0.465
2. dense + filters,0.725,0.554,0.490,0.686,0.824,0.611
3. BM25 + filters,0.784,0.618,0.588,0.784,0.882,0.701
4. hybrid (RRF),0.824,0.666,0.667,0.804,0.922,0.755
5. dense + rerank,0.863,0.689,0.686,0.882,0.902,0.769
6. hybrid + rerank,0.863,0.660,0.686,0.863,0.902,0.770
7. dense + rerank (fused),0.843,0.665,0.647,0.843,0.902,0.749


    Judged Hit@5 by type    


qtype,hand,image,table,text
variant,,,,
"1. dense, no filters",0.50,0.9,0.52,0.75
2. dense + filters,0.75,1.0,0.76,0.81
3. BM25 + filters,0.50,0.9,0.95,0.88
4. hybrid (RRF),0.75,1.0,0.95,0.88
5. dense + rerank,1.00,1.0,0.90,0.81
6. hybrid + rerank,0.75,1.0,0.95,0.81
7. dense + rerank (fused),1.00,1.0,0.86,0.88



Gold items judged NOT relevant: 24 | non-gold items judged relevant: 44

[q032] What is the primary name of the mobile banking application that anchors the digital ecosystem of AU Small Finance Bank in FY 2024-25?
  ref: AU 0101
  found: p120 → [text, FY2025 report, p.120] Leadership Indicators
1. Channels / platforms where information on products and services of the entity can be accessed (provide web link, if available).
AU Small Finance Bank provides comprehensive information on its products and services through multiple platforms. Cu

[q049] Which body is responsible for evaluating non-independent directors, the board as a whole, and the Chairman in AU Small Finance Bank during FY 2023-24?
  ref: Meeting of Independent Directors
  found: p91 → [text, FY2024 report, p.91] Independent Director's Meeting
As per the requirement of the Section 149(8) read with Schedule IV of Companies Act 2013, and Regulation 25 of SEBI (LODR) Regulations, a meeting of the Independent Directors of t

In [23]:
# Generating answers for the baseline and the best variant (cached)
from PIL import Image

ANS_MODEL = "gemini-3.5-flash-lite"       #same model for every setup
best = judged.groupby("variant")["j_hit@5"].mean().idxmax()
EVAL_VARIANTS = ["2. dense + filters", best]
print("Comparing:", EVAL_VARIANTS)

ANSWER_PROMPT = """You are a financial analyst assistant answering questions about company annual reports.
Rules:
- Use ONLY the sources provided below. Never use outside knowledge.
- Only report numbers that appear in the sources. You MAY compute a simple difference or
  % change between two figures stated in the sources, and must show the calculation.
  NEVER derive new quantities (e.g. back-calculating a total from a percentage).
- Quote exact numbers with their units (₹ crore, %, etc.) and fiscal year.
- Cite every fact like [AU Small Finance Bank FY2024, p.148].
- Include only points that directly answer the question; leave out tangential data.
- If the sources don't contain the answer, reply: "I couldn't find this in the provided reports."
- Keep it concise: a direct answer first, then 1-3 supporting points."""

def build_contents(q, ids):
    contents = [ANSWER_PROMPT, f"QUESTION: {q}", "SOURCES:"]
    for k, i in enumerate(ids, 1):
        h = docstore[i]
        src = f"[{h['company'].replace('_', ' ').title()} FY{h['year']}, p.{h['page']}]"
        if h["type"] == "image":
            contents += [f"Source {k} {src}: chart. Summary: {h.get('summary', '')}",
                         Image.open(h["image_path"])]
        elif h["type"] == "table":
            contents.append(f"Source {k} {src}: table\n{h['content'][:6000]}")
        else:
            contents.append(f"Source {k} {src}: text\n{h['content']}")
    return contents

def gemini_text(contents, model, retries=5):
    for a in range(retries):
        try:
            return client.models.generate_content(
                model=model, contents=contents,
                config=types.GenerateContentConfig(temperature=0)).text
        except Exception as ex:
            m = str(ex)
            if "503" in m or "UNAVAILABLE" in m:
                time.sleep(5 * 2 ** a)
            elif "429" in m or "RESOURCE_EXHAUSTED" in m:
                if "PerDay" in m:
                    raise RuntimeError(f"Daily quota reached for {model}, rerun tomorrow")
                time.sleep(30)
            else:
                raise
    return None

ANS_CACHE = EVAL_DIR / "answers_cache.json"
answers = json.loads(ANS_CACHE.read_text()) if ANS_CACHE.exists() else {}

for v in EVAL_VARIANTS:
    for row in tqdm(list(golden.itertuples()), desc=v):
        key = f"{v}|{row.qid}"
        if key in answers:
            continue
        ids = VARIANTS[v](row.question)[:8]
        text = gemini_text(build_contents(row.question, ids), ANS_MODEL)
        if text:
            answers[key] = text
            ANS_CACHE.write_text(json.dumps(answers, ensure_ascii=False))
        time.sleep(4)
print(f"{len(answers)} answers cached")

Comparing: ['2. dense + filters', '4. hybrid (RRF)']


2. dense + filters:   0%|          | 0/56 [00:00<?, ?it/s]

4. hybrid (RRF):   0%|          | 0/56 [00:00<?, ?it/s]

168 answers cached


In [24]:
# Grading the answers against the reference answers (short IDs so replies always match)
GRADE_PROMPT = """You grade answers from a financial Q&A system against reference answers.
For EACH item, compare SYSTEM ANSWER to REFERENCE ANSWER and give a verdict:
- "correct": the key facts and numbers match the reference (minor wording or rounding is fine)
- "partial": some key facts right, but some missing or wrong
- "incorrect": key facts wrong, OR it gave an answer when the reference is "NOT FOUND"
- "refused": the system said it couldn't find the answer
Return ONLY a JSON list: [{"id": "A1", "verdict": "correct"}, ...]"""

GRADE_CACHE = EVAL_DIR / "grades_cache.json"
grades = json.loads(GRADE_CACHE.read_text()) if GRADE_CACHE.exists() else {}
qa = golden.set_index("qid")

todo = [k for k in answers if k not in grades and k.split("|")[0] in EVAL_VARIANTS]
print(f"{len(todo)} answers to grade")
for b in tqdm(range(0, len(todo), 5)):
    ids = {f"A{j + 1}": k for j, k in enumerate(todo[b:b + 5])}
    items = [f"ID: {a}\nQUESTION: {qa.loc[k.split('|')[1], 'question']}\n"
             f"REFERENCE ANSWER: {qa.loc[k.split('|')[1], 'answer']}\nSYSTEM ANSWER: {answers[k]}"
             for a, k in ids.items()]
    out = gemini_json([GRADE_PROMPT] + items, model=JUDGE_MODEL)
    if isinstance(out, dict):
        out = out.get("items", [out])
    if not out:
        print(f"   no reply for batch {b // 5}, rerun this cell later"); continue
    for r in out:
        if isinstance(r, dict) and r.get("id") in ids:
            grades[ids[r["id"]]] = str(r.get("verdict", "")).strip().lower()
    GRADE_CACHE.write_text(json.dumps(grades))
    time.sleep(4)
print(f"{len(grades)} grades cached")

0 answers to grade


0it [00:00, ?it/s]

168 grades cached


In [25]:
# Building the final answer accuracy table
g = pd.DataFrame([{"variant": k.split("|")[0], "qid": k.split("|")[1], "verdict": v}
                  for k, v in grades.items() if k.split("|")[0] in EVAL_VARIANTS])
g["answerable"] = g.qid.map(qa["qtype"]) != "unanswerable"
print(g.groupby("variant").verdict.value_counts().unstack(fill_value=0))

def summarize(df):
    a, u = df[df.answerable], df[~df.answerable]
    return pd.Series({
        "correct": round((a.verdict == "correct").mean(), 3),
        "correct+partial": round(a.verdict.isin(["correct", "partial"]).mean(), 3),
        "wrongly refused": round((a.verdict == "refused").mean(), 3),
        "unanswerable refused": f"{(u.verdict == 'refused').sum()}/{len(u)}",
    })

final = g.groupby("variant")[["answerable", "verdict"]].apply(summarize)
display(final)
final.to_csv(EVAL_DIR / "answer_accuracy.csv")

verdict             correct  incorrect  partial  refused
variant                                                 
2. dense + filters       43          4        1        8
4. hybrid (RRF)          46          4        0        6


,correct,correct+partial,wrongly refused,unanswerable refused
variant,,,,
2. dense + filters,0.765,0.784,0.137,1/5
4. hybrid (RRF),0.804,0.804,0.118,0/5
